# Four-year Binance hourly direction dataset

This notebook replaces the active 15-minute task with a one-hour direction task. The older \`historical_binance_15m.ipynb\` remains as a completed small prototype and is not rerun here.

\`\`\`text
four years of historical BTCUSDT 1m klines
                 ↓
hourly decisions at HH:00
                 ↓
features known before decision + next non-overlapping 60m label
                 ↓
audit CSV + model-ready CSV + chronological 70/15/15 split report
\`\`\`

The historical REST source has interval timestamps but not the original client receipt time. This dataset therefore records the \`interval_complete_assumption\`; it is not receipt-time verified. This notebook does not use Polymarket, Chainlink, GPU computation, trading logic, or live execution.


## Operating rules

- Colab is stateless. Git stores code; Drive stores raw CSVs, checkpoints, outputs, and reports.
- The downloader checkpoints one UTC day at a time and skips only verified daily files.
- The builder checkpoints one target day at a time and skips only verified 24-row daily files.
- A decision at \`10:00\` uses completed history through the \`09:59\` close and predicts \`10:00\` through \`10:59\`.
- Features use only completed 1-minute bars before the decision time.
- Target prices and labels remain in the audit view but never enter model feature columns.
- Split by whole UTC dates before counting usable rows; never randomize rows.
- The current complete target range ends at \`2026-08-15\`; the current partial day is excluded.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from collections import Counter
from datetime import date

REPOSITORY = 'https://github.com/matahariramadhan/tradingbot-data.git'
REVISION = '925e4d9f9a94a7ffb9f777caafbbe7badde337d1'
PROJECT_DIR = Path('/content/tradingbot_v2')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    assert (PROJECT_DIR / '.git').is_dir(), f'not a Git checkout: {PROJECT_DIR}'

subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '.[training]'], cwd=PROJECT_DIR, check=True)
sys.path.insert(0, str(PROJECT_DIR))

print('repository:', REPOSITORY)
print('revision:', subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True
).strip())


In [ ]:
from importlib.metadata import version as distribution_version
import tradingbot_data

assert distribution_version('tradingbot-data') == '0.10.0'
assert tradingbot_data.__version__ == '0.10.0'
help_text = subprocess.check_output(['tradingbot-data', '--help'], text=True)
assert 'historical-download' in help_text
assert 'historical-hourly' in help_text
print('distribution version:', distribution_version('tradingbot-data'))
print('package version:', tradingbot_data.__version__)
print(help_text)


## 1. Mount Drive and define the durable artifact contract

The raw range includes one warm-up day, \`2022-08-15\`. The target range is the latest four complete UTC years: \`2022-08-16\` through \`2026-08-15\` inclusive. The split uses 70%/15%/15% of the 1461 target days.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONTROL_DIR = Path('/content/drive/MyDrive/tradingbot-data-audit')
HIST_RAW_DIR = CONTROL_DIR / 'historical-binance-1m-4y-v1'
HIST_DATASET_DIR = CONTROL_DIR / 'historical-binance-hourly-4y-v1'
DOWNLOAD_CHECKPOINT = CONTROL_DIR / 'historical-binance-1m-4y-download-v1.json'
DOWNLOAD_REPORT = CONTROL_DIR / 'historical-binance-1m-4y-download-report-v1.json'
DATASET_CHECKPOINT = CONTROL_DIR / 'historical-binance-hourly-4y-build-v1.json'
DATASET_REPORT = CONTROL_DIR / 'historical-binance-hourly-4y-report-v1.json'
SPLIT_REPORT = CONTROL_DIR / 'historical-binance-hourly-4y-split-v1.json'
AUDIT_OUTPUT = HIST_DATASET_DIR / 'dataset-audit-v1.csv'
MODEL_OUTPUT = HIST_DATASET_DIR / 'model-ready-v1.csv'

RAW_START = '2022-08-15'
RAW_END = '2026-08-16'
TARGET_START = '2022-08-16'
TARGET_END = '2026-08-16'
TRAIN_END = '2025-06-04'
VALIDATION_END = '2026-01-09'

target_days = (date.fromisoformat(TARGET_END) - date.fromisoformat(TARGET_START)).days
train_days = (date.fromisoformat(TRAIN_END) - date.fromisoformat(TARGET_START)).days
validation_days = (date.fromisoformat(VALIDATION_END) - date.fromisoformat(TRAIN_END)).days
holdout_days = (date.fromisoformat(TARGET_END) - date.fromisoformat(VALIDATION_END)).days
assert (target_days, train_days, validation_days, holdout_days) == (1461, 1023, 219, 219)

print('raw output:', HIST_RAW_DIR)
print('target days:', target_days)
print('split days:', train_days, validation_days, holdout_days)
print('split ratio:', [round(value / target_days, 4) for value in (train_days, validation_days, holdout_days)])


## 2. Download the independent historical source

This is the long-running cell. It may make thousands of API requests. If Colab stops, rerun it: completed daily files are checkpointed on Drive and will be skipped. The interrupted day is the maximum unit that may repeat.


In [ ]:
subprocess.run([
    'tradingbot-data', 'historical-download',
    '--symbol', 'BTCUSDT',
    '--start-date', RAW_START,
    '--end-date', RAW_END,
    '--output-dir', str(HIST_RAW_DIR),
    '--checkpoint', str(DOWNLOAD_CHECKPOINT),
    '--report', str(DOWNLOAD_REPORT),
    '--request-delay-seconds', '0.1',
], cwd=PROJECT_DIR, check=True)


In [ ]:
assert DOWNLOAD_REPORT.is_file(), f'missing download report: {DOWNLOAD_REPORT}'
download_report = json.loads(DOWNLOAD_REPORT.read_text(encoding='utf-8'))
assert download_report['status'] == 'completed'
assert download_report['config']['interval'] == '1m'
assert download_report['config']['start_date'] == RAW_START
assert download_report['config']['end_date_exclusive'] == RAW_END
assert download_report['receipt_time_available'] is False
assert download_report['availability_policy'] == 'interval_complete_assumption'
assert download_report['totals']['days'] == 1462
print(json.dumps(download_report['totals'], indent=2))


## 3. Build the hourly dataset and split report

The builder emits 24 hourly rows per target day. It writes target details only to the audit view and verifies final output shape, unique keys, chronological order, and zero partition overlap.


In [ ]:
subprocess.run([
    'tradingbot-data', 'historical-hourly',
    '--raw-dir', str(HIST_RAW_DIR),
    '--download-report', str(DOWNLOAD_REPORT),
    '--target-start', TARGET_START,
    '--target-end', TARGET_END,
    '--train-end', TRAIN_END,
    '--validation-end', VALIDATION_END,
    '--output-dir', str(HIST_DATASET_DIR),
    '--checkpoint', str(DATASET_CHECKPOINT),
    '--report', str(DATASET_REPORT),
    '--split-report', str(SPLIT_REPORT),
], cwd=PROJECT_DIR, check=True)


In [ ]:
assert DATASET_REPORT.is_file(), f'missing dataset report: {DATASET_REPORT}'
assert SPLIT_REPORT.is_file(), f'missing split report: {SPLIT_REPORT}'
dataset_report = json.loads(DATASET_REPORT.read_text(encoding='utf-8'))
split_report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
assert dataset_report['status'] == 'completed'
assert split_report['status'] == 'completed'
assert dataset_report['config']['horizon_minutes'] == 60
assert dataset_report['config']['cadence_minutes'] == 60
assert split_report['verification']['train_validation_holdout_overlap_keys'] == 0
assert split_report['verification']['model_keys_unique'] is True
assert split_report['verification']['chronological_model_keys'] is True
print(json.dumps(dataset_report['totals'], indent=2))
print(json.dumps(split_report['totals'], indent=2))


In [ ]:
with MODEL_OUTPUT.open(newline='', encoding='utf-8') as source:
    model_reader = csv.DictReader(source)
    model_rows = list(model_reader)
with AUDIT_OUTPUT.open(newline='', encoding='utf-8') as source:
    audit_reader = csv.DictReader(source)
    audit_columns = audit_reader.fieldnames

assert model_reader.fieldnames is not None
assert 'label' in model_reader.fieldnames
assert 'target_start_price' not in model_reader.fieldnames
assert 'target_end_price' not in model_reader.fieldnames
assert 'target_return_60m' not in model_reader.fieldnames
assert audit_columns is not None and 'target_return_60m' in audit_columns
assert len(model_rows) == dataset_report['totals']['model_rows_usable']
print('audit columns:', audit_columns)
print('model columns:', model_reader.fieldnames)
print('model-ready rows:', len(model_rows))


## 4. Human-readable review

These plots reload durable CSV artifacts. They do not train a model and do not use future target prices as input features.


In [ ]:
import matplotlib.pyplot as plt

feature_columns = [
    'return_1m', 'return_5m', 'return_15m', 'return_30m', 'return_60m',
    'volatility_5m', 'volatility_15m', 'volatility_60m',
    'volume_ratio_5m', 'candle_body_5m', 'high_low_range_5m',
    'distance_ma_15', 'ma_slope_15', 'rsi_14',
]
label_counts = Counter(row['label'] for row in model_rows)
with AUDIT_OUTPUT.open(newline='', encoding='utf-8') as source:
    audit_rows = list(csv.DictReader(source))
target_returns = [
    float(row['target_return_60m'])
    for row in audit_rows
    if row['target_valid'] == 'true'
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(sorted(label_counts), [label_counts[label] for label in sorted(label_counts)])
axes[0].set_title('Hourly label counts')
axes[0].set_ylabel('rows')
axes[1].hist(target_returns, bins=80)
axes[1].set_title('Future one-hour returns')
axes[1].set_xlabel('return')
for field in feature_columns[:6]:
    axes[2].plot(sorted(float(row[field]) for row in model_rows), label=field)
axes[2].set_title('Selected sorted feature values')
axes[2].set_xlabel('sorted row index')
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.show()


## What this gate means

At this point we have an auditable four-year hourly supervised-learning table, not evidence of predictive skill. The next lesson is to inspect usable-row coverage, class balance, invalid-row reasons, and the chronological report before fitting a simple baseline.
